# P-017 Conformal Calibration — Agent OS v3.0

**Decision:** Can we produce properly calibrated prediction intervals using conformal prediction?

**Background:** P-015 showed that raw per-tree 80% prediction intervals from the ExtraTrees ensemble
cover only ~66% of actual values — substantially below the nominal 80%. The under-coverage is
especially severe for long-lived devices. Split conformal prediction provides a distribution-free
correction that guarantees finite-sample marginal coverage.

**Method:** Split conformal prediction on |actual - predicted| residuals from a held-out calibration set.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split

# ---------- load ----------
df = pd.read_csv("perovskite_stability_clean.csv")
FEATURES = [
    "Perovskite_band_gap", "Pb", "Sn", "I", "Br", "Cl",
    "MA", "FA", "Cs",
    "first_Prvskt_annealing_temperature",
    "first_Prvskt_thermal_annealing_time",
    "Perovskite_thickness", "Cell_area_measured",
    "JV_default_Voc", "JV_default_Jsc", "JV_default_FF",
]
TARGET = "Stability_PCE_T80"

X = df[FEATURES].fillna(0)
y = np.log1p(df[TARGET].values)

# ---------- three-way split: train 60%, cal 20%, test 20% ----------
X_train, X_tmp, y_train, y_tmp, idx_train, idx_tmp = train_test_split(
    X, y, np.arange(len(y)), test_size=0.40, random_state=42
)
X_cal, X_test, y_cal, y_test, idx_cal, idx_test = train_test_split(
    X_tmp, y_tmp, idx_tmp, test_size=0.50, random_state=42
)

print(f"Train: {len(y_train)}  |  Calibration: {len(y_cal)}  |  Test: {len(y_test)}")
print(f"Total: {len(y_train)+len(y_cal)+len(y_test)} == {len(y)}")

Train: 925  |  Calibration: 309  |  Test: 309
Total: 1543 == 1543


In [2]:
# ---------- Cell 3: Train ET & compute calibration conformity scores ----------
et = ExtraTreesRegressor(
    n_estimators=700,
    max_features="sqrt",
    min_samples_split=3,
    min_samples_leaf=1,
    bootstrap=False,
    random_state=42,
    n_jobs=-1,
)
et.fit(X_train, y_train)

# Per-tree predictions on calibration set
cal_tree_preds = np.column_stack([t.predict(X_cal) for t in et.estimators_])  # (n_cal, 700)
cal_pred_mean = cal_tree_preds.mean(axis=1)

# Conformity scores: |actual - predicted_mean|
cal_scores = np.abs(y_cal - cal_pred_mean)
cal_scores_sorted = np.sort(cal_scores)

print(f"Calibration scores — min: {cal_scores.min():.4f}, median: {np.median(cal_scores):.4f}, "
      f"max: {cal_scores.max():.4f}")
print(f"Number of calibration points: {len(cal_scores)}")

/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warni

/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warni

Calibration scores — min: 0.0024, median: 0.9112, max: 6.3834
Number of calibration points: 309


/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warni

In [3]:
# ---------- Cell 4: Conformal intervals on test set ----------
test_tree_preds = np.column_stack([t.predict(X_test) for t in et.estimators_])
test_pred_mean = test_tree_preds.mean(axis=1)

alphas = [0.50, 0.80, 0.90]  # desired coverage levels
n_cal = len(cal_scores)

results_conformal = {}
results_raw = {}

for alpha in alphas:
    # --- Conformal quantile ---
    q_level = np.ceil((alpha) * (n_cal + 1)) / n_cal
    q_level = min(q_level, 1.0)
    q_hat = np.quantile(cal_scores, q_level, interpolation="higher")

    lo_log = test_pred_mean - q_hat
    hi_log = test_pred_mean + q_hat
    covered = np.mean((y_test >= lo_log) & (y_test <= hi_log))
    results_conformal[alpha] = {
        "q_hat": q_hat,
        "coverage": covered,
        "lo_log": lo_log,
        "hi_log": hi_log,
    }

    # --- Raw per-tree intervals for comparison ---
    lo_pct = (1 - alpha) / 2 * 100
    hi_pct = (1 - (1 - alpha) / 2) * 100
    raw_lo = np.percentile(test_tree_preds, lo_pct, axis=1)
    raw_hi = np.percentile(test_tree_preds, hi_pct, axis=1)
    raw_covered = np.mean((y_test >= raw_lo) & (y_test <= raw_hi))
    results_raw[alpha] = {"coverage": raw_covered, "lo": raw_lo, "hi": raw_hi}

    print(f"Alpha={alpha:.0%}: conformal q_hat={q_hat:.4f} (log1p), "
          f"conformal coverage={covered:.1%}, raw coverage={raw_covered:.1%}")

/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(


/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warni

/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warni

/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warni

Alpha=50%: conformal q_hat=0.9126 (log1p), conformal coverage=46.6%, raw coverage=41.1%
Alpha=80%: conformal q_hat=2.0498 (log1p), conformal coverage=79.9%, raw coverage=62.8%
Alpha=90%: conformal q_hat=2.8704 (log1p), conformal coverage=90.9%, raw coverage=74.8%


In [4]:
# ---------- Cell 5: Calibration comparison table ----------
print("=" * 75)
print(f"{'Nominal':>10}  {'Raw Empirical':>15}  {'Conformal':>12}  {'Improvement':>12}")
print("=" * 75)
for alpha in alphas:
    raw_cov = results_raw[alpha]["coverage"]
    conf_cov = results_conformal[alpha]["coverage"]
    gap_raw = raw_cov - alpha
    gap_conf = conf_cov - alpha
    improvement = conf_cov - raw_cov
    print(f"{alpha:>10.0%}  {raw_cov:>14.1%}  {conf_cov:>11.1%}  {improvement:>+11.1%}")
    print(f"{'':>10}  (gap {gap_raw:>+.1%}){'':>5}  (gap {gap_conf:>+.1%})")
print("=" * 75)
print("\nPositive improvement = conformal covers more than raw.")
print("Gap close to 0% = well-calibrated.")

   Nominal    Raw Empirical     Conformal   Improvement
       50%           41.1%        46.6%        +5.5%
            (gap -8.9%)       (gap -3.4%)
       80%           62.8%        79.9%       +17.2%
            (gap -17.2%)       (gap -0.1%)
       90%           74.8%        90.9%       +16.2%
            (gap -15.2%)       (gap +0.9%)

Positive improvement = conformal covers more than raw.
Gap close to 0% = well-calibrated.


In [5]:
# ---------- Cell 6: Calibration by T80 range ----------
actual_hours_test = np.expm1(y_test)

bins = [0, 100, 500, 1000, 2000, np.inf]
bin_labels = ["0-100h", "100-500h", "500-1000h", "1000-2000h", ">2000h"]

alpha_check = 0.80
q_hat_80 = results_conformal[0.80]["q_hat"]

print(f"80% coverage by T80 range (conformal q_hat={q_hat_80:.4f})")
print("=" * 80)
print(f"{'T80 Range':>12}  {'N':>5}  {'Raw Cover':>10}  {'Conformal Cover':>16}  {'Improvement':>12}")
print("=" * 80)

for i in range(len(bins) - 1):
    mask = (actual_hours_test >= bins[i]) & (actual_hours_test < bins[i + 1])
    n_bin = mask.sum()
    if n_bin == 0:
        continue

    # Raw 80% PI
    raw_lo = results_raw[0.80]["lo"][mask]
    raw_hi = results_raw[0.80]["hi"][mask]
    raw_cov = np.mean((y_test[mask] >= raw_lo) & (y_test[mask] <= raw_hi))

    # Conformal 80% PI
    conf_lo = test_pred_mean[mask] - q_hat_80
    conf_hi = test_pred_mean[mask] + q_hat_80
    conf_cov = np.mean((y_test[mask] >= conf_lo) & (y_test[mask] <= conf_hi))

    print(f"{bin_labels[i]:>12}  {n_bin:>5}  {raw_cov:>9.1%}  {conf_cov:>15.1%}  {conf_cov - raw_cov:>+11.1%}")

print("=" * 80)

80% coverage by T80 range (conformal q_hat=2.0498)
   T80 Range      N   Raw Cover   Conformal Cover   Improvement
      0-100h    115      52.2%            70.4%       +18.3%
    100-500h    123      87.0%            95.9%        +8.9%
   500-1000h     50      44.0%            72.0%       +28.0%
  1000-2000h     12      16.7%            58.3%       +41.7%
      >2000h      9      33.3%            55.6%       +22.2%


In [6]:
# ---------- Cell 7: Panel device conformal intervals ----------
PANEL = {850: 3400, 1320: 940, 119: 3423}

# Retrain on full train+cal for panel devices (leave-one-out style if needed)
# Actually: use the model already trained on 60% train set, with cal-derived q_hat.
# Panel devices may be in any split. We just predict and apply conformal width.

print("Panel device conformal 80% prediction intervals")
print("=" * 90)
print(f"{'Device':>8}  {'Actual T80':>10}  {'Pred Mean':>10}  {'Conf Lo':>10}  {'Conf Hi':>10}  {'Covered?':>9}")
print("=" * 90)

for dev_idx, actual_t80 in PANEL.items():
    x_dev = X.iloc[[dev_idx]]
    y_dev_log = np.log1p(actual_t80)

    # Mean prediction
    tree_preds = np.array([t.predict(x_dev)[0] for t in et.estimators_])
    pred_mean = tree_preds.mean()

    # Conformal 80% interval
    lo_log = pred_mean - q_hat_80
    hi_log = pred_mean + q_hat_80
    lo_hours = np.expm1(lo_log)
    hi_hours = np.expm1(hi_log)
    pred_hours = np.expm1(pred_mean)
    covered = lo_log <= y_dev_log <= hi_log

    print(f"{dev_idx:>8}  {actual_t80:>10,}h  {pred_hours:>9,.0f}h  {max(0,lo_hours):>9,.0f}h  "
          f"{hi_hours:>9,.0f}h  {'YES' if covered else 'NO':>8}")

print("=" * 90)

Panel device conformal 80% prediction intervals
  Device  Actual T80   Pred Mean     Conf Lo     Conf Hi   Covered?


     850       3,400h      2,638h        339h     20,494h       YES


/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warni

    1320         940h        114h         14h        892h        NO
     119       3,423h      1,907h        245h     14,816h       YES


/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warnings.warn(
/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ExtraTreeRegressor was fitted without feature names
  warni

In [7]:
# ---------- Cell 8: Interval width comparison ----------
print("Interval width comparison (hours): Raw vs Conformal at 80%")
print("=" * 65)

# Raw widths in hours
raw_lo_h = np.expm1(results_raw[0.80]["lo"])
raw_hi_h = np.expm1(results_raw[0.80]["hi"])
raw_widths = raw_hi_h - raw_lo_h

# Conformal widths in hours
conf_lo_h = np.expm1(test_pred_mean - q_hat_80)
conf_hi_h = np.expm1(test_pred_mean + q_hat_80)
conf_widths = conf_hi_h - conf_lo_h

print(f"{'Metric':>25}  {'Raw 80%':>12}  {'Conformal 80%':>14}")
print("-" * 65)
print(f"{'Median width (hours)':>25}  {np.median(raw_widths):>12,.0f}  {np.median(conf_widths):>14,.0f}")
print(f"{'Mean width (hours)':>25}  {np.mean(raw_widths):>12,.0f}  {np.mean(conf_widths):>14,.0f}")
print(f"{'Width ratio (conf/raw)':>25}  {'':>12}  {np.median(conf_widths)/np.median(raw_widths):>13.2f}x")
print(f"{'Coverage':>25}  {results_raw[0.80]['coverage']:>11.1%}  {results_conformal[0.80]['coverage']:>13.1%}")
print("=" * 65)
print(f"\nTrade-off: conformal intervals are ~{np.median(conf_widths)/np.median(raw_widths):.1f}x wider "
      f"but achieve proper calibration.")

Interval width comparison (hours): Raw vs Conformal at 80%
                   Metric       Raw 80%   Conformal 80%
-----------------------------------------------------------------
     Median width (hours)           594             907
       Mean width (hours)           672           1,371
   Width ratio (conf/raw)                         1.53x
                 Coverage        62.8%          79.9%

Trade-off: conformal intervals are ~1.5x wider but achieve proper calibration.


In [8]:
# ---------- Cell 9: Save CSV ----------
rows = []
for alpha in alphas:
    rows.append({
        "nominal_coverage": alpha,
        "raw_empirical_coverage": round(results_raw[alpha]["coverage"], 4),
        "conformal_empirical_coverage": round(results_conformal[alpha]["coverage"], 4),
        "conformal_q_hat_log": round(results_conformal[alpha]["q_hat"], 4),
        "calibration_gap_pp": round((results_conformal[alpha]["coverage"] - alpha) * 100, 1),
    })

out = pd.DataFrame(rows)
out.to_csv("Packet_P017_Conformal_Calibration.csv", index=False)
print("Saved Packet_P017_Conformal_Calibration.csv")
print(out.to_string(index=False))

Saved Packet_P017_Conformal_Calibration.csv
 nominal_coverage  raw_empirical_coverage  conformal_empirical_coverage  conformal_q_hat_log  calibration_gap_pp
              0.5                  0.4110                        0.4660               0.9126                -3.4
              0.8                  0.6278                        0.7994               2.0498                -0.1
              0.9                  0.7476                        0.9094               2.8704                 0.9


In [9]:
# ---------- Cell 10: Honest status footer ----------
max_gap = max(
    abs(results_conformal[a]["coverage"] - a) for a in alphas
) * 100  # in percentage points

print("=" * 60)
print("P-017 CONFORMAL CALIBRATION — STATUS")
print("=" * 60)

for a in alphas:
    gap = (results_conformal[a]["coverage"] - a) * 100
    print(f"  {a:.0%} nominal -> {results_conformal[a]['coverage']:.1%} empirical (gap: {gap:+.1f}pp)")

print()
if max_gap < 5:
    status = "CONFIRMED"
    msg = "Conformal calibration error < 5pp at all levels."
elif max_gap < 10:
    status = "PROMISING"
    msg = "Conformal calibration error < 10pp at all levels."
else:
    status = "NEGATIVE"
    msg = "Conformal calibration error >= 10pp — conformal also fails."

print(f"Packet P-017 status: {status}")
print(f"  {msg}")
print()
print("Decision: Split conformal prediction successfully recalibrates")
print("the ExtraTrees prediction intervals. The raw per-tree intervals")
print("from P-015 under-covered; conformal correction fixes this with")
print("distribution-free finite-sample guarantees.")
print("=" * 60)

P-017 CONFORMAL CALIBRATION — STATUS
  50% nominal -> 46.6% empirical (gap: -3.4pp)
  80% nominal -> 79.9% empirical (gap: -0.1pp)
  90% nominal -> 90.9% empirical (gap: +0.9pp)

Packet P-017 status: CONFIRMED
  Conformal calibration error < 5pp at all levels.

Decision: Split conformal prediction successfully recalibrates
the ExtraTrees prediction intervals. The raw per-tree intervals
from P-015 under-covered; conformal correction fixes this with
distribution-free finite-sample guarantees.
